# Day 053 — Exercise 5: The Frontend Gateway

**What you'll build:** `AIAppClient` — the frontend's typed gateway to the backend. It wraps an injected HTTP client and exposes `health()`, `chat(message)`, `templates()`, and `render(name, topic)`, each returning plain data the UI renders.

**Why it matters:** This is the seam between the two halves of your app. The Streamlit UI holds one `AIAppClient` and never touches HTTP directly — the same thin-shell-over-logic pattern as Days 51 and 52, now spanning the network. Because the client is injected, the tests drive it against an in-process backend; `frontend.py` drives it against a live `httpx.Client`.

## Provided: Setup + Backend + check_health + post_chat + request_json

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import httpx
import ollama


# ---- The AI backend (built on Day 52 — provided here) ----
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    reply: str
    model: str


class HealthResponse(BaseModel):
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app


def check_health(client) -> bool:
    """Ping the backend's GET /health through an injected HTTP client.

    Returns True only if the request succeeds with 200 AND status == 'ok'.
    Any exception (backend down, connection refused) -> False, never raises.
    The `client` is duck-typed: an httpx.Client in production, a TestClient in
    tests — both expose .get / .post / .request.
    """
    try:
        resp = client.get('/health')
        return resp.status_code == 200 and resp.json().get('status') == 'ok'
    except Exception:
        return False


def post_chat(client, message: str, temperature: float = 0.7) -> dict:
    """POST /chat honouring the JSON contract {message, temperature}.

    Returns the parsed {reply, model} on 200. On a non-200 status returns
    {'error': ..., 'status': code}; on a connection failure returns
    {'error': ...}. The frontend never sees a raw exception.
    """
    try:
        resp = client.post('/chat', json={'message': message, 'temperature': temperature})
    except Exception as e:
        return {'error': f'request failed: {e}'}
    if resp.status_code != 200:
        return {'error': f'backend returned {resp.status_code}', 'status': resp.status_code}
    return resp.json()


def request_json(client, method: str, path: str, payload: dict = None) -> dict:
    """Call the backend and normalise EVERY outcome into one envelope:

        {'ok': bool, 'status': int | None, 'data': dict | None, 'error': str | None}

    - success (2xx):        ok=True,  status=code, data=json
    - error status (4xx/5xx): ok=False, status=code, error='HTTP <code>'
    - connection failure:   ok=False, status=None, error='connection error: ...'

    One shape for the whole frontend to branch on — no scattered try/except.
    """
    try:
        resp = client.request(method, path, json=payload)
    except Exception as e:
        return {'ok': False, 'status': None, 'data': None,
                'error': f'connection error: {e}'}
    ok = 200 <= resp.status_code < 300
    try:
        data = resp.json()
    except Exception:
        data = None
    return {
        'ok':     ok,
        'status': resp.status_code,
        'data':   data if ok else None,
        'error':  None if ok else f'HTTP {resp.status_code}',
    }

## Your Implementation

In [ ]:
class AIAppClient:
    """Typed gateway to the AI backend. Wraps an injected HTTP client."""

    def __init__(self, client):
        # TODO: self.client = client
        pass

    def health(self) -> bool:
        # TODO: return check_health(self.client)
        pass

    def chat(self, message: str, temperature: float = 0.7) -> dict:
        # TODO: return post_chat(self.client, message, temperature)
        pass

    def templates(self) -> list:
        # TODO: env = request_json(self.client, 'GET', '/templates')
        # TODO: return env['data']['templates'] if env['ok'] else []
        pass

    def render(self, name: str, topic: str, temperature: float = 0.7) -> dict:
        # TODO: env = request_json(self.client, 'POST', f'/render/{name}',
        #                          {'message': topic, 'temperature': temperature})
        # TODO: return env['data'] if env['ok'] else {'error': env['error']}
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    backend = TestClient(build_api())

    # Check 1: gateway exposes the four methods
    try:
        assert 'AIAppClient' in globals()
        for m in ('health', 'chat', 'templates', 'render'):
            assert hasattr(AIAppClient, m), f'missing method: {m}'
        api = AIAppClient(backend)
        passed += 1; print('✅ Check 1: AIAppClient has health/chat/templates/render')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: health() -> True against the live in-process backend
    try:
        assert api.health() is True, 'health() should be True'
        passed += 1; print('✅ Check 2: api.health() is True')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: chat() returns a reply (Ollama)
    try:
        out = api.chat('Say hello in three words.')
        assert isinstance(out, dict) and len(out.get('reply', '')) > 0, f'bad chat: {out}'
        passed += 1; print('✅ Check 3: api.chat() returns a reply')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: templates() lists the names
    try:
        names = api.templates()
        assert isinstance(names, list) and 'summary' in names, f'bad templates: {names}'
        passed += 1; print('✅ Check 4: api.templates() lists names')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: render() works for a known name, errors for an unknown one
    try:
        ok = api.render('summary', 'Python')
        assert len(ok.get('reply', '')) > 0, f'expected a reply: {ok}'
        missing = api.render('does-not-exist', 'x')
        assert 'error' in missing, f'expected an error for unknown template: {missing}'
        passed += 1; print('✅ Check 5: render() renders known / errors unknown')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class AIAppClient:
    """The frontend's typed gateway to the AI backend.

    Wraps an injected HTTP client (httpx.Client in production, TestClient in
    tests) so the exact same code runs against a live server or in-process.
    Every method returns plain data the UI can render — no HTTP details leak out.
    """

    def __init__(self, client):
        self.client = client

    def health(self) -> bool:
        return check_health(self.client)

    def chat(self, message: str, temperature: float = 0.7) -> dict:
        return post_chat(self.client, message, temperature)

    def templates(self) -> list:
        env = request_json(self.client, 'GET', '/templates')
        return env['data']['templates'] if env['ok'] else []

    def render(self, name: str, topic: str, temperature: float = 0.7) -> dict:
        env = request_json(self.client, 'POST', f'/render/{name}',
                           {'message': topic, 'temperature': temperature})
        return env['data'] if env['ok'] else {'error': env['error']}
```

**Why this works:** `AIAppClient` composes the three functions you built into one object with a clean method per backend route. It never exposes HTTP details — `health()` gives a bool, `chat()` gives a dict, `templates()` gives a list — so the Streamlit UI code stays about *display*, not networking. Injecting the client keeps it testable: the checks use `TestClient(build_api())`; `frontend.py` uses `httpx.Client(base_url='http://localhost:8000')`.
</details>